# 04 — Self-Assertion KGL Expansion

This notebook performs FastText-based synonym expansion and concept expansion
for the frozen Self-Assertion Seed Set v1.

No purification is performed in this notebook.

In [5]:
from pathlib import Path

import pandas as pd
import numpy as np
import gensim.downloader as api

In [6]:
PROJECT_ROOT = Path("..")

SEED_FILE = (
    PROJECT_ROOT
    / "data"
    / "seed_candidates"
    / "self_assertion_seed_set_v1.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "expansion"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

Current working directory:
/Users/catherinexie/Documents/kgl-mockproject/notebooks


In [3]:
PROJECT_ROOT = Path("..")

SEED_FILE = (
    PROJECT_ROOT
    / "data"
    / "seed_candidates"
    / "self_assertion_seed_set_v1.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "expansion"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Seed file exists:", SEED_FILE.exists())
print("Output directory exists:", OUTPUT_DIR.exists())

Project root: /Users/catherinexie/Documents/kgl-mockproject
Seed file exists: True
Output directory exists: True


In [7]:
seed_df = pd.read_csv(SEED_FILE)

print("Number of frozen seeds:", len(seed_df))
print(seed_df["seed"].tolist())

Number of frozen seeds: 14
['urge', 'propose', 'reject', 'advocate', 'reaffirm', 'insist', 'declare', 'resist', 'assert', 'defend', 'self-assertion', 'intend', 'ready', 'exercise']


In [8]:
print(seed_df)

              seed                     facet     source final_decision
0             urge           advance_outcome   original           keep
1          propose           advance_outcome   original           keep
2           reject         resist_constraint   original           keep
3         advocate           advance_outcome   original           keep
4         reaffirm         maintain_position   original           keep
5           insist         maintain_position   original           keep
6          declare            claim_position   original           keep
7           resist         resist_constraint   original           keep
8           assert            claim_position   original  theory_anchor
9           defend      defend_self_relevant   original  theory_anchor
10  self-assertion           construct_label   original  theory_anchor
11          intend    commit_intended_action  gap_added           keep
12           ready    commit_intended_action  gap_added           keep
13    

In [9]:
MODEL_NAME = "fasttext-wiki-news-subwords-300"

ft = api.load(MODEL_NAME)

[==================================================] 100.0% 958.5/958.4MB downloaded


In [10]:
print(type(ft))
print("Vector dimensions:", ft.vector_size)

<class 'gensim.models.keyedvectors.KeyedVectors'>
Vector dimensions: 300


In [11]:
seed_vector_check = []

for seed in seed_df["seed"]:
    try:
        vec = ft.get_vector(seed)
        available = True
        norm = np.linalg.norm(vec)
    except KeyError:
        available = False
        norm = np.nan

    seed_vector_check.append({
        "seed": seed,
        "vector_available": available,
        "vector_norm": norm
    })

seed_vector_check = pd.DataFrame(seed_vector_check)

seed_vector_check

,seed,vector_available,vector_norm
0,urge,True,1.288498
1,propose,True,0.800554
2,reject,True,0.966094
3,advocate,True,0.788096
4,reaffirm,True,0.884402
5,insist,True,0.845252
6,declare,True,0.838490
7,resist,True,0.989003
8,assert,True,0.873927
9,defend,True,0.968208


In [12]:
seed_vector_check["vector_available"].value_counts()

vector_available
True    14
Name: count, dtype: int64

In [13]:
seed_vectors = {}

for seed in seed_df["seed"]:
    vec = ft.get_vector(seed)
    vec = vec / np.linalg.norm(vec)
    seed_vectors[seed] = vec

In [14]:
for seed in list(seed_vectors.keys())[:5]:
    print(seed, np.linalg.norm(seed_vectors[seed]))

urge 1.0
propose 1.0
reject 1.0
advocate 1.0
reaffirm 1.0


In [15]:
SYNONYM_THRESHOLD = 0.75
CONCEPT_THRESHOLD = 0.45

In [16]:
def synonym_expand(
    model,
    seeds,
    similarity_threshold=0.75,
    topn=500
):
    rows = []

    for seed in seeds:
        neighbors = model.most_similar(
            positive=[seed],
            topn=topn
        )

        for word, similarity in neighbors:
            if similarity >= similarity_threshold:
                rows.append({
                    "seed": seed,
                    "expanded_term": word,
                    "similarity": similarity,
                    "expansion_type": "synonym"
                })

    return pd.DataFrame(rows)

In [17]:
synonym_expansion = synonym_expand(
    model=ft,
    seeds=seed_df["seed"].tolist(),
    similarity_threshold=SYNONYM_THRESHOLD,
    topn=500
)

print("Total synonym-expansion rows:", len(synonym_expansion))

synonym_expansion.head(30)

Total synonym-expansion rows: 51


,seed,expanded_term,similarity,expansion_type
0,urge,urging,0.790209,synonym
1,urge,urges,0.789152,synonym
2,urge,urged,0.775632,synonym
3,urge,implore,0.773136,synonym
4,propose,proposing,0.818100,synonym
5,propose,re-propose,0.805075,synonym
6,propose,repropose,0.797981,synonym
7,propose,suggest,0.783833,synonym
8,propose,counter-propose,0.781766,synonym
9,propose,proposes,0.777562,synonym


In [18]:
synonym_counts = (
    synonym_expansion
    .groupby("seed")
    .size()
    .reset_index(name="n_expanded_terms")
    .sort_values(
        "n_expanded_terms",
        ascending=False
    )
)

synonym_counts

,seed,n_expanded_terms
7,propose,7
8,reaffirm,7
0,advocate,6
4,exercise,5
1,assert,4
2,declare,4
9,reject,4
12,urge,4
5,insist,3
6,intend,2


In [19]:
pd.set_option("display.max_rows", 200)

synonym_expansion.sort_values(
    ["seed", "similarity"],
    ascending=[True, False]
)

,seed,expanded_term,similarity,expansion_type
15,advocate,advocates,0.831214,synonym
16,advocate,advocating,0.795685,synonym
17,advocate,advocation,0.761888,synonym
18,advocate,advocated,0.761289,synonym
19,advocate,advocator,0.756424,synonym
20,advocate,self-advocate,0.750883,synonym
37,assert,asserting,0.779048,synonym
38,assert,re-assert,0.756543,synonym
39,assert,claim,0.753320,synonym
40,assert,reassert,0.751403,synonym


In [20]:
expanded_seeds = set(synonym_expansion["seed"])

zero_synonym_seeds = [
    seed
    for seed in seed_df["seed"]
    if seed not in expanded_seeds
]

print("Seeds with zero synonym expansions above .75:")
print(zero_synonym_seeds)

Seeds with zero synonym expansions above .75:
['ready']


In [21]:
SYNONYM_FILE = (
    OUTPUT_DIR
    / "self_assertion_synonym_expansion_primary.csv"
)

synonym_expansion.to_csv(
    SYNONYM_FILE,
    index=False
)

print(SYNONYM_FILE)

../data/expansion/self_assertion_synonym_expansion_primary.csv


In [22]:
seed_matrix = np.vstack(
    [
        seed_vectors[seed]
        for seed in seed_df["seed"]
    ]
)

concept_centroid = seed_matrix.mean(axis=0)

concept_centroid = (
    concept_centroid
    / np.linalg.norm(concept_centroid)
)

print("Centroid dimensions:", concept_centroid.shape)
print("Centroid norm:", np.linalg.norm(concept_centroid))

Centroid dimensions: (300,)
Centroid norm: 1.0


In [23]:
CONCEPT_THRESHOLD = 0.45
CONCEPT_TOPN = 10000

In [24]:
concept_neighbors = ft.similar_by_vector(
    concept_centroid,
    topn=CONCEPT_TOPN
)

In [25]:
concept_expansion = pd.DataFrame(
    concept_neighbors,
    columns=["expanded_term", "similarity"]
)

concept_expansion = concept_expansion[
    concept_expansion["similarity"] >= CONCEPT_THRESHOLD
].copy()

concept_expansion["expansion_type"] = "concept"

concept_expansion = (
    concept_expansion
    .sort_values(
        "similarity",
        ascending=False
    )
    .reset_index(drop=True)
)

concept_expansion

,expanded_term,similarity,expansion_type
0,assert,0.797689,concept
1,urge,0.789836,concept
2,reject,0.781349,concept
3,propose,0.774361,concept
4,defend,0.772548,concept
...,...,...,...
9995,survive,0.557673,concept
9996,soliloquize,0.557673,concept
9997,carte-blanche,0.557669,concept
9998,clarfication,0.557664,concept


In [26]:
lowest_returned_similarity = concept_neighbors[-1][1]

print(
    "Similarity of the 10,000th nearest term:",
    lowest_returned_similarity
)

Similarity of the 10,000th nearest term: 0.5576624870300293


In [29]:
CONCEPT_TOPN = 100000

concept_neighbors = ft.similar_by_vector(
    concept_centroid,
    topn=CONCEPT_TOPN
)

In [32]:
lowest_returned_similarity = concept_neighbors[-1][1]

print(
    "Similarity of the 100,000th nearest term:",
    lowest_returned_similarity
)

Similarity of the 100,000th nearest term: 0.4218613803386688


In [33]:
CONCEPT_THRESHOLD = 0.45

concept_expansion = pd.DataFrame(
    concept_neighbors,
    columns=["expanded_term", "similarity"]
)

concept_expansion = concept_expansion[
    concept_expansion["similarity"] >= CONCEPT_THRESHOLD
].copy()

concept_expansion["expansion_type"] = "concept"

concept_expansion = (
    concept_expansion
    .sort_values(
        "similarity",
        ascending=False
    )
    .reset_index(drop=True)
)

In [34]:
print(
    "Number of concept-expanded terms:",
    len(concept_expansion)
)

Number of concept-expanded terms: 69110


In [35]:
concept_expansion.head(50)

,expanded_term,similarity,expansion_type
0,assert,0.797689,concept
1,urge,0.789836,concept
2,reject,0.781349,concept
3,propose,0.774361,concept
4,defend,0.772548,concept
5,declare,0.770948,concept
6,denounce,0.769357,concept
7,compell,0.763614,concept
8,temporize,0.763041,concept
9,counter-propose,0.761597,concept


In [36]:
concept_expansion.tail(20)

,expanded_term,similarity,expansion_type
69090,notices---,0.450014,concept
69091,cold-shouldering,0.450014,concept
69092,Islamicate,0.450014,concept
69093,head-turning,0.450014,concept
69094,repito,0.450013,concept
69095,go-tos,0.450013,concept
69096,slack-jawed,0.450012,concept
69097,up-field,0.450011,concept
69098,samay,0.450011,concept
69099,examples,0.450010,concept


In [37]:
CONCEPT_FILE = (
    OUTPUT_DIR
    / "self_assertion_concept_expansion_primary.csv"
)

concept_expansion.to_csv(
    CONCEPT_FILE,
    index=False
)

print(CONCEPT_FILE)

../data/expansion/self_assertion_concept_expansion_primary.csv


In [38]:
raw_seed_matrix = np.vstack(
    [
        ft.get_vector(seed)
        for seed in seed_df["seed"]
    ]
)

raw_concept_centroid = raw_seed_matrix.mean(axis=0)

print("Centroid dimensions:", raw_concept_centroid.shape)
print("Centroid norm:", np.linalg.norm(raw_concept_centroid))

Centroid dimensions: (300,)
Centroid norm: 0.6557597


In [39]:
raw_centroid_neighbors = ft.similar_by_vector(
    raw_concept_centroid,
    topn=100000
)

In [40]:
print(
    "Similarity of the 100,000th nearest term:",
    raw_centroid_neighbors[-1][1]
)

Similarity of the 100,000th nearest term: 0.41644158959388733


In [41]:
for threshold in [0.40, 0.45, 0.50]:
    n = sum(
        similarity >= threshold
        for _, similarity in raw_centroid_neighbors
    )
    
    print(
        f"Threshold {threshold:.2f}: {n:,} terms"
    )

Threshold 0.40: 100,000 terms
Threshold 0.45: 63,791 terms
Threshold 0.50: 28,572 terms


In [42]:
for threshold in [0.55, 0.60, 0.65, 0.70]:
    n = sum(
        similarity >= threshold
        for _, similarity in raw_centroid_neighbors
    )
    
    print(
        f"Threshold {threshold:.2f}: {n:,} terms"
    )

Threshold 0.55: 10,713 terms
Threshold 0.60: 3,435 terms
Threshold 0.65: 841 terms
Threshold 0.70: 146 terms


In [43]:
def show_terms_above_threshold(
    neighbors,
    threshold,
    n_show=50
):
    filtered = [
        (word, sim)
        for word, sim in neighbors
        if sim >= threshold
    ]
    
    print(
        f"Threshold {threshold:.2f}: "
        f"{len(filtered):,} terms"
    )
    
    return pd.DataFrame(
        filtered[:n_show],
        columns=["term", "similarity"]
    )

In [44]:
show_terms_above_threshold(
    raw_centroid_neighbors,
    0.45
)

Threshold 0.45: 63,791 terms


,term,similarity
0,urge,0.807748
1,assert,0.787626
2,reject,0.785834
3,propose,0.777625
4,defend,0.775598
5,denounce,0.772007
6,declare,0.768392
7,compell,0.767535
8,temporize,0.764389
9,accept,0.758967


In [45]:
show_terms_above_threshold(
    raw_centroid_neighbors,
    0.55
)

Threshold 0.55: 10,713 terms


,term,similarity
0,urge,0.807748
1,assert,0.787626
2,reject,0.785834
3,propose,0.777625
4,defend,0.775598
5,denounce,0.772007
6,declare,0.768392
7,compell,0.767535
8,temporize,0.764389
9,accept,0.758967


In [46]:
show_terms_above_threshold(
    raw_centroid_neighbors,
    0.60
)

Threshold 0.60: 3,435 terms


,term,similarity
0,urge,0.807748
1,assert,0.787626
2,reject,0.785834
3,propose,0.777625
4,defend,0.775598
5,denounce,0.772007
6,declare,0.768392
7,compell,0.767535
8,temporize,0.764389
9,accept,0.758967


In [47]:
show_terms_above_threshold(
    raw_centroid_neighbors,
    0.65
)

Threshold 0.65: 841 terms


,term,similarity
0,urge,0.807748
1,assert,0.787626
2,reject,0.785834
3,propose,0.777625
4,defend,0.775598
5,denounce,0.772007
6,declare,0.768392
7,compell,0.767535
8,temporize,0.764389
9,accept,0.758967


In [48]:
def show_bottom_terms_above_threshold(
    neighbors,
    threshold,
    n_show=20
):
    filtered = [
        (word, sim)
        for word, sim in neighbors
        if sim >= threshold
    ]
    
    return pd.DataFrame(
        filtered[-n_show:],
        columns=["term", "similarity"]
    )

In [49]:
show_bottom_terms_above_threshold(
    raw_centroid_neighbors,
    0.60
)

,term,similarity
0,soom,0.600257
1,misspeak,0.600246
2,strugling,0.600234
3,agreee,0.600228
4,chack,0.600219
5,dispell,0.600218
6,reast,0.600199
7,contridicting,0.600197
8,certaint,0.600179
9,invitiation,0.600179


In [50]:
show_bottom_terms_above_threshold(
    raw_centroid_neighbors,
    0.65,
    n_show=50
)

,term,similarity
0,expection,0.651922
1,acknowlege,0.651873
2,abase,0.651838
3,denegrate,0.651794
4,negate,0.651768
5,individualise,0.651752
6,rewite,0.651682
7,re-collect,0.651677
8,suppose,0.651634
9,mortify,0.651630


In [51]:
show_bottom_terms_above_threshold(
    raw_centroid_neighbors,
    0.70,
    n_show=50
)

,term,similarity
0,minimalize,0.713155
1,expouse,0.713013
2,admit,0.712357
3,agress,0.711505
4,re-claim,0.711379
5,minimalise,0.710698
6,submit,0.710528
7,support,0.709948
8,pursue,0.709723
9,disaprove,0.709386


In [52]:
concept_070 = pd.DataFrame(
    [
        (word, similarity)
        for word, similarity in raw_centroid_neighbors
        if similarity >= 0.70
    ],
    columns=["expanded_term", "similarity"]
)

pd.set_option("display.max_rows", 200)

concept_070

,expanded_term,similarity
0,urge,0.807748
1,assert,0.787626
2,reject,0.785834
3,propose,0.777625
4,defend,0.775598
5,denounce,0.772007
6,declare,0.768392
7,compell,0.767535
8,temporize,0.764389
9,accept,0.758967


In [53]:
CONCEPT_THRESHOLD = 0.45

concept_expansion = pd.DataFrame(
    raw_centroid_neighbors,
    columns=["expanded_term", "similarity"]
)

concept_expansion = concept_expansion[
    concept_expansion["similarity"] >= CONCEPT_THRESHOLD
].copy()

concept_expansion["expansion_type"] = "concept"

concept_expansion = (
    concept_expansion
    .sort_values("similarity", ascending=False)
    .reset_index(drop=True)
)

print("Raw concept-expansion terms:", len(concept_expansion))

Raw concept-expansion terms: 63791


In [54]:
CONCEPT_FILE = (
    OUTPUT_DIR
    / "self_assertion_concept_expansion_primary.csv"
)

concept_expansion.to_csv(
    CONCEPT_FILE,
    index=False
)

print(CONCEPT_FILE)

../data/expansion/self_assertion_concept_expansion_primary.csv


In [55]:
seed_rows = pd.DataFrame({
    "term": seed_df["seed"],
    "weight": 1.0,
    "source": "seed"
})

seed_rows.head()

,term,weight,source
0,urge,1.0,seed
1,propose,1.0,seed
2,reject,1.0,seed
3,advocate,1.0,seed
4,reaffirm,1.0,seed


In [56]:
synonym_rows = synonym_expansion.rename(
    columns={
        "expanded_term": "term",
        "similarity": "weight"
    }
)[["term", "weight", "seed"]].copy()

synonym_rows["source"] = (
    "synonym:" + synonym_rows["seed"].astype(str)
)

synonym_rows = synonym_rows[
    ["term", "weight", "source"]
]

synonym_rows.head()

,term,weight,source
0,urging,0.790209,synonym:urge
1,urges,0.789152,synonym:urge
2,urged,0.775632,synonym:urge
3,implore,0.773136,synonym:urge
4,proposing,0.818100,synonym:propose


In [57]:
concept_rows = concept_expansion.rename(
    columns={
        "expanded_term": "term",
        "similarity": "weight"
    }
)[["term", "weight"]].copy()

concept_rows["source"] = "concept"

concept_rows.head()

,term,weight,source
0,urge,0.807748,concept
1,assert,0.787626,concept
2,reject,0.785834,concept
3,propose,0.777625,concept
4,defend,0.775598,concept


In [58]:
raw_long = pd.concat(
    [
        seed_rows,
        synonym_rows,
        concept_rows
    ],
    ignore_index=True
)

print("Total rows before resolving duplicates:", len(raw_long))

raw_long.head()

Total rows before resolving duplicates: 63856


,term,weight,source
0,urge,1.0,seed
1,propose,1.0,seed
2,reject,1.0,seed
3,advocate,1.0,seed
4,reaffirm,1.0,seed


In [59]:
raw_union = (
    raw_long
    .groupby("term", as_index=False)
    .agg(
        weight=("weight", "max"),
        sources=(
            "source",
            lambda x: "; ".join(sorted(set(x)))
        )
    )
)

In [60]:
raw_union = raw_union.sort_values(
    ["weight", "term"],
    ascending=[False, True]
).reset_index(drop=True)

In [61]:
print(
    "Unique terms in raw expanded lexicon:",
    len(raw_union)
)

raw_union.head(30)

Unique terms in raw expanded lexicon: 63791


,term,weight,sources
0,advocate,1.000000,concept; seed
1,assert,1.000000,concept; seed
2,declare,1.000000,concept; seed
3,defend,1.000000,concept; seed
4,exercise,1.000000,concept; seed
5,insist,1.000000,concept; seed
6,intend,1.000000,concept; seed
7,propose,1.000000,concept; seed
8,ready,1.000000,concept; seed
9,reaffirm,1.000000,concept; seed


In [62]:
check_terms = [
    "assert",
    "urge",
    "ready",
    "accept",
    "implore",
    "self-assertion"
]

raw_union[
    raw_union["term"].isin(check_terms)
]

,term,weight,sources
1,assert,1.000000,concept; seed
8,ready,1.000000,concept; seed
12,self-assertion,1.000000,concept; seed
13,urge,1.000000,concept; seed
29,accept,0.800865,concept; synonym:reject
47,implore,0.773136,concept; synonym:urge


In [63]:
RAW_EXPANDED_FILE = (
    OUTPUT_DIR
    / "self_assertion_expanded_raw_primary.csv"
)

raw_union.to_csv(
    RAW_EXPANDED_FILE,
    index=False
)

print(RAW_EXPANDED_FILE)

../data/expansion/self_assertion_expanded_raw_primary.csv
